# Exploration of California Vehicle Miles Traveled Data
We will visualize California Vehicle Miles Traveled (VMT) data using the example of San Francisco. This provides a template for student analysis of the potential climate benefits of infill housing development for another city or county in California, following approaches similar to Subin (2024[a](https://ternercenter.berkeley.edu/research-and-policy/role-of-new-housing-in-reducing-climate-pollution/), [b](https://ternercenter.berkeley.edu/blog/how-much-can-new-housing-contribute-to-state-climate-action/)) and Subin et al ([2026](https://ternercenter.berkeley.edu/blog/projected-transportation-impacts-of-california-housing-plans/)).

The source data consists of preliminary VMT estimates developed by [Chatman et al 2025](https://escholarship.org/uc/item/99j4s0bp) for the California Air Resources Board, provided by CARB to Zack Subin in July 2026 for educational use. This data file is available to students via the course Canvas site and should be downloaded to the `inputdata` folder. Note: a version of these estimates is now publicly available as of Sep 2026, posted by CARB for public comment at the [California Climate Investments page](https://www.caclimateinvestments.ca.gov/public-participation-opportunities), integrated into the [Draft Affordable Housing Benefits Calculator Tool](https://ww2.arb.ca.gov/sites/default/files/auction-proceeds/DRAFT_AffordableHousing_CalculatorTool_090826.xlsx).

We will use the neighborhood ([census block group](https://en.wikipedia.org/wiki/Census_block_group)) estimates of VMT _variation_ from the state average here. Each row consists of a census block group [FIPS](https://www.census.gov/programs-surveys/geography/guidance/geo-identifiers.html) code and the average per-capita VMT of residents living within the census block group, relative to the state average, based on a statistical model trained on year-2017 data from the National Household Travel Survey ([NHTS](https://nhts.ornl.gov/)).

Steps:

1. Load data, clean, confirm the distribution is as expected, and then convert from a relative to absolute VMT per capita by adding the statewide-mean VMT.
2. Overlay with a standard census geographic shapefile for California census block groups, excerpt for San Francisco, and visualize.
3. Overlay with population data from the most recently available American Community Survey (ACS) year.
4. Explore descriptive statistics useful for climate analysis: weighted averages and 10th percentile.

In [1]:
# Setup
import numpy as np # Core numerical and array operations
import pandas as pd # Data table and analysis operations
import matplotlib.pyplot as plt # Plotting; syntax and appearance similar to Matlab
import pathlib # file handling robust to switching between Mac, PC, and Linux platforms
import geopandas as gpd # GIS functionality added to pandas. We will use lightly here to make a map.

# Step 1: VMT data cleaning

In [2]:
cwd = pathlib.Path().cwd() # Current working directory
cwd

PosixPath('/Users/zsubin/Code/usf-engy680-urban-energy-fall2026/hw2_OptionA_VMT')

In [3]:
inputdir = cwd / 'inputdata' # pathlib abstracts the Mac vs. PC syntax difference using '/' not as a string
# but an operator here (see e.g. https://en.wikipedia.org/wiki/Operator_overloading)
inputdir

PosixPath('/Users/zsubin/Code/usf-engy680-urban-energy-fall2026/hw2_OptionA_VMT/inputdata')

Make sure that the path is correct and matches the `inputdata` folder inside the `hw2_OptionA_VMT` subfolder. (Depending on how you launched this notebook, your
current working directory could be a level or two up.)

Make sure you have downloaded the preliminary data file available via the course Canvas page (you must be a member of the course), and saved it into `inputdata`.

    🤓 You could always put this file elsewhere, but this demonstrates using relative rather than absolute paths which are helpful when running the same code on multiple computers. If you put it elsewhere, just make sure it's somewhere outside of git tracking.

In [4]:
datafile = inputdir / 'CARB-UCB-CA_VMT_Estimates_DataVintage_Oct2025_PRELIMINARY_FOR_STUDENT_USE.csv'
# hard-coded input file name to match the file you'll find on Canvas.

# Read the data into a pandas Dataframe
data = pd.read_csv(datafile)
data.head()

,1,GEOID,RetDens_bg_z,DensBusStops_one_z,NetLoadDens_GF18_two_z,JobsWi45_transit_to_car_ratio,JobsWithin45_Car_z,StreetDens_GF18_two_z,cbsa_IE,cbsa_LA_OC,VMT_state,VMT_state_trunc,VMT_diff_state,VMT_CBSA,VMT_CBSA_trunc,VMT_diff_CBSA
0,2,60259400003,0.130,0.362,3.634,-0.190,0.735,0.3702,0.0,0.0,5.040,5.040,6.370,5.040,5.040,3.273
1,3,60250108002,0.133,0.362,3.500,0.000,0.698,0.4617,0.0,0.0,5.155,5.155,6.486,5.155,5.155,3.388
2,4,60250108001,0.133,0.362,3.511,0.000,0.692,0.4865,0.0,0.0,5.185,5.185,6.515,5.185,5.185,3.418
3,5,60250111003,0.133,0.362,3.513,0.000,0.681,0.4965,0.0,0.0,5.185,5.185,6.516,5.185,5.185,3.418
4,6,60730100132,-0.025,0.042,-2.598,-0.764,-0.048,-0.2850,0.0,0.0,-3.677,-3.677,-2.346,-3.677,-3.677,-2.384


The first two columns are apparently dummy indices which we will drop.

The two columns we are interested in are `GEOID` which represents 12-digit FIPS codes for census block groups,
and `VMT_diff_state` which represents VMT per person, per day differences from the state average. (You will often obtain data with extraneous columns that need to be trimmed.)

First, we need to clean the `GEOID` column: you'll notice these are 11 digits rather than 12. The reason is
that California's 2-digit state FIPS code is "06". Pushing this data through Excel to a csv and then into python
has converted it to integers and dropped the leading zero. In order to match with other geographic data, we'll need
to restore the leading zero. We'll do this by applying a function to each element in the column.

In [5]:
data['GEOID'] = data['GEOID'].map(str).str.zfill(12)
data.head()

,1,GEOID,RetDens_bg_z,DensBusStops_one_z,NetLoadDens_GF18_two_z,JobsWi45_transit_to_car_ratio,JobsWithin45_Car_z,StreetDens_GF18_two_z,cbsa_IE,cbsa_LA_OC,VMT_state,VMT_state_trunc,VMT_diff_state,VMT_CBSA,VMT_CBSA_trunc,VMT_diff_CBSA
0,2,060259400003,0.130,0.362,3.634,-0.190,0.735,0.3702,0.0,0.0,5.040,5.040,6.370,5.040,5.040,3.273
1,3,060250108002,0.133,0.362,3.500,0.000,0.698,0.4617,0.0,0.0,5.155,5.155,6.486,5.155,5.155,3.388
2,4,060250108001,0.133,0.362,3.511,0.000,0.692,0.4865,0.0,0.0,5.185,5.185,6.515,5.185,5.185,3.418
3,5,060250111003,0.133,0.362,3.513,0.000,0.681,0.4965,0.0,0.0,5.185,5.185,6.516,5.185,5.185,3.418
4,6,060730100132,-0.025,0.042,-2.598,-0.764,-0.048,-0.2850,0.0,0.0,-3.677,-3.677,-2.346,-3.677,-3.677,-2.384


This syntax is obscure but handy. This means first convert the column from integers to strings,
then apply the `str.zfill` [series member function](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.zfill.html) to the column (a single pandas column is a series).

This method is robust: it will behave as desired even if the data were already strings, or if you loaded data from a mix of states where some
had 12-digit codes.

Now we'll drop the unwanted columns, set GEOID to be the index, and look at the VMT data.

In [8]:
data_cleaned = data.set_index('GEOID')[['VMT_diff_state']]
# set index and keep only a list of columns with one member (hence the double brackets)
data_cleaned.head()

,VMT_diff_state
GEOID,
060259400003,6.370
060250108002,6.486
060250108001,6.515
060250111003,6.516
060730100132,-2.346


Now let's look at the data! Pandas has a handy [describe](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.describe.html) function here which provides count, mean, standard deviation, minimum, maximum, and quartiles.

In [9]:
data_cleaned.describe()

,VMT_diff_state
count,23190.000000
mean,-0.256459
std,4.770563
min,-14.135000
25%,-2.704750
50%,0.303000
75%,2.936000
max,10.676000


We can confirm this is an expected distribution for daily, per-capita VMT deviations from the state mean.
You'll notice that the mean is not quite zero-- that's because it's designed to be zero when weighted by population,
but we are just using each census block group as a data point, and block groups have some variation in population.

Likewise, you'll notice the median ("50%") is higher than the mean. This is not expected as VMT should be right-skewed (nowhere can drive less than zero, but a few rural places drive quite a lot). Hopefully this will improve
after we incorporate population.

But first, we would like to visualize _absolute_ VMT, not just relative VMT to the state average. This source does not provide the state average, so we will have to assume one.

It is easy to find statewide VMT from Caltrans or [FHWA](https://www.fhwa.dot.gov/policyinformation/statistics/2024/vm2.cfm). However, these combine light-duty vehicles (LDVs) with medium- and heavy-duty trucks, buses, etc. A decent assumption is that 90% of VMT is from LDV passenger travel (you can confirm this by exploring other FHWA tables), but we can get a more precise estimate for California from CARB's [EMFAC](https://emfac.arb.ca.gov/emissions-inventory/) database.

Let's use calendar year 2017 to match the Chatman et al primary data source, choose statewide on-road emission rates, and select vehicle categories LDA, LDT1, LDT2, and MCY to include light-duty autos, trucks (i.e. SUVs and pickups), and motorcycles. (These category acronyms are defined if you click the "?" icon on the website.)

![](./EMFAC_Screenshot1.png)
![](./EMFAC_Screenshot2.png)

This yields a table with one row for each vehicle category. Quickly summing the "Total VMT" column in Excel yields 716 million miles per day. Let's divide by California's 2017 population [from ACS via the state Department of Finance](https://dof.ca.gov/reports/demographic-reports/american-community-survey/#ACS2017x1) of 39.5 million residents to estimate the statewide mean VMT.

In [10]:
VMT_state = 716 / 39.5 # million VMT/day / million people = #VMT/person/day
VMT_state

18.126582278481013

Let's confirm that's reasonable compared with HW1. Multiplying by 365 yields 6,610 miles per person, per year.
That is in the ballpark since we previously assumed that each _vehicle_ drives 10,000 miles per year, and there are more people than vehicles. Plus, CA is a bit below the US average.

Nevertheless, this estimate is a tad lower than I expected comparing with recent Replica and FHWA estimates used in [Strangeway and Subin 2025](https://ternercenter.berkeley.edu/research-and-policy/aligning-housing-with-climate-goals-the-importance-of-measuring-vmt/). It is a different year and it's within 10%, so we'll go with it.

**Caution ⚠️** We have mixed two different VMT metrics here, which you can see discussed in the above reference: EMFAC is measuring the VMT driven on roads within the boundaries of the state, while the neighborhood residential VMT estimates are measuring the VMT driven by _residents_ of the state, wherever that mileage occurs.
This distinction is very important at the local scale and makes it much more challenging to obtain good VMT data for housing policy. However, at the statewide scale the distinction is very slight: only affected by visitors driving to the state or residents taking road trips outside the state (we are excluding freight which would be the vast majority of interstate trips).

OK let's add this to the census-block group deviations to get estimates of block-group VMT.

In [11]:
vmt = data_cleaned + VMT_state
vmt = vmt.rename({'VMT_diff_state': 'VMTpc'}, axis=1) # rename, using VMTpc as a shorthand for VMT per capita [per day]
vmt.describe()

,VMTpc
count,23190.000000
mean,17.870123
std,4.770563
min,3.991582
25%,15.421832
50%,18.429582
75%,21.062582
max,28.802582


I suspect these estimates are biased low at the high end, with a lack of right skew -- and consistent with the NHTS-derived dataset LATCH being found to have similar deficiency in [Subin et al 2024](https://journal-buildingscities.org/articles/10.5334/bc.434) -- but they are the best publicly available option so we'll just keep that in mind. (The estimates were developed to assess potential VMT savings from affordable housing projects, so it is reasonable they focus more on the low end of the distribution rather than precisely estimating potential _increases_ in VMT from affordable housing projects in rural locations, which are anyway unlikely to be developed and funded.)

For students who analyze outer suburban and rural places, note that resulting estimates of climate benefits from building housing may be biased high.

# Step 2: visualization